# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artasam/Machine-Learning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains the first ML model for **Lane 2 (Refresh / Content Opportunity Scoring)**
and compares it against the rule-based baseline from ML-07 on the **same data, same split,
same metric**.

Following `training-honest-models/SKILL.md`:
1. **Method choice** — why this method fits our "which ones first?" question
2. **Split design** — GroupKFold by `client_hash_id` (honest, no client memorization)
3. **Train + compare** — one comparison table: baseline vs model(s), same metric, base rate
4. **Errors and interpretation** — where is the model wrong? what does it lean on?

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `training-honest-models` + `flyrank/flyrank-data` for this task.

In [ ]:
%pip -q install duckdb huggingface_hub requests scikit-learn lightgbm

In [1]:
import os, getpass

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

print('Token loaded:', 'YES ✅' if HF_TOKEN else 'NO ❌')

Token loaded: YES ✅


In [2]:
import requests

headers = {'Authorization': f'Bearer {HF_TOKEN}'}
r = requests.get('https://huggingface.co/api/whoami-v2', headers=headers, timeout=10)
if r.status_code == 200:
    print(f'✅ Token valid. Account: {r.json().get("name", "?")}')
else:
    raise RuntimeError(f'❌ Token rejected ({r.status_code}).')

r2 = requests.get('https://huggingface.co/api/datasets/FlyRank/internship-warehouse',
                   headers=headers, timeout=10)
if r2.status_code == 200:
    print('✅ Gate accepted.')
elif r2.status_code == 403:
    raise RuntimeError('❌ Gate not accepted.')
else:
    print(f'⚠️ Status {r2.status_code}')

✅ Token valid. Account: Artasam-Khan
✅ Gate accepted.


In [3]:
import duckdb
import pandas as pd
import numpy as np
import json, pathlib

SEED = 42  # Fixed seed for reproducibility (skill requirement)
np.random.seed(SEED)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

n = con.sql(f"SELECT COUNT(*) FROM read_parquet('{REL}/dim_clients.parquet')").fetchone()[0]
print(f'✅ DuckDB connected. dim_clients: {n} rows.')
print(f'Random seed: {SEED}')

✅ DuckDB connected. dim_clients: 104 rows.
Random seed: 42


In [4]:
# Build feature vector — SAME query as ML-07 baseline.
# Feature window: Mar 1-15. Label window: Mar 16-31. Strict temporal isolation.

df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,

        -- Features (Mar 1-15 only)
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS prev_impressions,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS prev_clicks,
        AVG(CASE WHEN report_date <= '2026-03-15' AND gsc_avg_position > 0
            THEN gsc_avg_position END)                                              AS prev_avg_position,
        SUM(CASE WHEN report_date <= '2026-03-15' AND gsc_impressions > 0
            THEN 1 ELSE 0 END)                                                      AS prev_days_active,

        -- Label component (Mar 16-31) — NOT a feature
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END)  AS imp_last15

    FROM {FACT_MARCH}
    GROUP BY content_hash_id, client_hash_id
    HAVING prev_impressions >= 50
""").df()

# Derived features (feature window only)
df['log_prev_impressions'] = np.log1p(df['prev_impressions'])
df['prev_ctr'] = df['prev_clicks'] / (df['prev_impressions'] + 1)
df['prev_avg_position'] = df['prev_avg_position'].fillna(50.0)

# Proxy label — for evaluation only, NEVER enters features
df['is_declining'] = (df['imp_last15'] < 0.8 * df['prev_impressions']).astype(int)

print(f"Feature vector: {len(df):,} pages")
print(f"Base rate: {df['is_declining'].mean():.1%} declining")
print(f"Clients: {df['client_hash_id'].nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector: 92,548 pages
Base rate: 28.6% declining
Clients: 40


---

## 1. Method choice and why

**Our question shape:** *"Which pages should a content team review first?"* — this is a
**"which first?" ranking** problem.

The `training-honest-models/SKILL.md` toolkit says:
> *"which first?" ranking → any classifier's probability, evaluated at precision@K*

**Method progression (simple → stronger):**

| Model | Why |
|---|---|
| **Logistic Regression** | Readable baseline — can we beat the rule with a linear fit? |
| **Random Forest** | Captures non-linear interactions (position × volume) that ML-06 found |
| **LightGBM** | Gradient-boosted trees — strongest learner for tabular data with interactions |

The skill also says: *"Simplicity is a feature: a depth-2 decision tree you can print and
read teaches more than an opaque model 2 points stronger."* So we start with Logistic
Regression, then add complexity only if it earns an improvement.

**Features used (all from Mar 1-15 feature window only):**
- `log_prev_impressions` — log-scaled traffic volume (handles heavy tails)
- `prev_ctr` — click-through rate
- `prev_avg_position` — average search position
- `prev_days_active` — days with ≥1 impression in 15-day window

---

## 2. Split design

**Why GroupShuffleSplit by `client_hash_id`?**

The `hunting-leakage-and-validating/SKILL.md` says:
> *"Grouped split (GroupKFold by the entity that repeats — client, site, user): rows from
> one group share hidden character; a random split lets the model memorize the group and
> fake skill. The honest question is 'does it work on a group it never saw?'"*

Pages from the same client share writing style, topic niche, and domain authority. A random
split lets the model memorize client-specific patterns and report inflated scores. GroupSplit
ensures the test set contains ENTIRELY unseen clients.

We also report the random-split number alongside the grouped-split number — the GAP between
them is itself a finding about memorization.

In [28]:
from sklearn.model_selection import GroupShuffleSplit

# Features — ONLY from Mar 1-15 window. NO label-derived columns.
FEATURE_COLS = ['log_prev_impressions', 'prev_ctr', 'prev_avg_position', 'prev_days_active']

X = df[FEATURE_COLS].values
y = df['is_declining'].values
groups = df['client_hash_id'].values

# Grouped split: 80% train, 20% test — grouped by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_clients = set(groups[train_idx])
test_clients  = set(groups[test_idx])
overlap = train_clients & test_clients

print(f"GROUPED SPLIT (by client_hash_id)")
print(f"="  * 50)
print(f"Train: {len(train_idx):,} pages ({len(train_clients)} clients)")
print(f"Test:  {len(test_idx):,} pages ({len(test_clients)} clients)")
print(f"Client overlap: {len(overlap)} (must be 0)")
print(f"Train base rate: {y_train.mean():.1%}")
print(f"Test base rate:  {y_test.mean():.1%}")

GROUPED SPLIT (by client_hash_id)
Train: 70,017 pages (32 clients)
Test:  22,531 pages (8 clients)
Client overlap: 0 (must be 0)
Train base rate: 26.0%
Test base rate:  37.0%


---

## 3. Train + compare vs my baseline

*Same data, same metric, same split as the Week-4 baseline. Show the table.*

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print('⚠️ LightGBM not available, skipping.')

# ======================================================================
# PRECISION@K — same function as ML-07 baseline
# ======================================================================
def precision_at_k(scores, labels, k):
    """Of the top K items by score, what fraction are actually declining?"""
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# ======================================================================
# BASELINE RULE SCORE (reproduced from ML-07 — same formula, test set only)
# ======================================================================
POSITION_PENALTY = {
    'top_3': 0.3, 'page_1': 0.5, 'striking': 1.5,
    'page_3_5': 1.2, 'deep': 0.8, 'no_data': 0.5
}

def position_tier(pos):
    if pd.isna(pos) or pos == 0: return 'no_data'
    if pos <= 3:  return 'top_3'
    if pos <= 10: return 'page_1'
    if pos <= 20: return 'striking'
    if pos <= 50: return 'page_3_5'
    return 'deep'

test_df = df.iloc[test_idx].copy()
test_df['pos_tier'] = test_df['prev_avg_position'].apply(position_tier)
test_df['baseline_score'] = (
    np.log1p(test_df['prev_impressions'])
    * test_df['pos_tier'].map(POSITION_PENALTY)
    * (1 - np.minimum(test_df['prev_ctr'] * 20, 0.5))
)

baseline_scores_test = test_df['baseline_score'].values

print('Baseline rule reproduced on test set. ✅')

Baseline rule reproduced on test set. ✅


In [30]:
# ======================================================================
# TRAIN ALL MODELS
# ======================================================================

# Prepare DataFrames to preserve feature names for tree models
X_train_df = pd.DataFrame(X_train, columns=FEATURE_COLS)
X_test_df = pd.DataFrame(X_test, columns=FEATURE_COLS)

# Model 1: Logistic Regression (needs scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
print('✅ Logistic Regression trained.')

# Model 2: Random Forest
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=50,
    random_state=SEED, n_jobs=-1
)
rf.fit(X_train_df, y_train)
rf_probs = rf.predict_proba(X_test_df)[:, 1]
print('✅ Random Forest trained.')

# Model 3: LightGBM
if HAS_LGBM:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbose=-1
    )
    lgb_model.fit(X_train_df, y_train)
    lgb_probs = lgb_model.predict_proba(X_test_df)[:, 1]
    print('✅ LightGBM trained.')

✅ Logistic Regression trained.
✅ Random Forest trained.
✅ LightGBM trained.


In [31]:
# ======================================================================
# THE COMPARISON TABLE (non-negotiable per SKILL.md)
# Same data, same split, same metric. Base rate printed alongside.
# ======================================================================

k_values = [10, 20, 50, 100, 200]
base_rate_test = y_test.mean()

# Build all models' scores
models = {
    'Rule Baseline (ML-07)': baseline_scores_test,
    'Logistic Regression':   lr_probs,
    'Random Forest':         rf_probs,
}
if HAS_LGBM:
    models['LightGBM'] = lgb_probs

print("COMPARISON TABLE — Precision@K (Grouped Split, Test Set Only)")
print("=" * 85)
header = f"{'Model':<28}"
for k in k_values:
    header += f" {'P@'+str(k):>8}"
print(header)
print("-" * 85)

# Base rate row
base_row = f"{'Random (base rate)':<28}"
for k in k_values:
    base_row += f" {base_rate_test:>8.3f}"
print(base_row)

print("-" * 85)

results = {}
for name, scores in models.items():
    row = f"{name:<28}"
    results[name] = {}
    for k in k_values:
        p = precision_at_k(scores, y_test, k)
        row += f" {p:>8.3f}"
        results[name][f'P@{k}'] = round(p, 4)
    print(row)

print("=" * 85)
print(f"\nTest set: {len(y_test):,} pages | Base rate: {base_rate_test:.1%}")
print(f"Split: GroupShuffleSplit by client_hash_id (0 client overlap)")
print(f"Seed: {SEED}")

COMPARISON TABLE — Precision@K (Grouped Split, Test Set Only)
Model                            P@10     P@20     P@50    P@100    P@200
-------------------------------------------------------------------------------------
Random (base rate)              0.370    0.370    0.370    0.370    0.370
-------------------------------------------------------------------------------------
Rule Baseline (ML-07)           0.600    0.750    0.640    0.600    0.540
Logistic Regression             0.200    0.150    0.200    0.220    0.235
Random Forest                   0.200    0.250    0.320    0.390    0.450
LightGBM                        0.400    0.400    0.440    0.490    0.515

Test set: 22,531 pages | Base rate: 37.0%
Split: GroupShuffleSplit by client_hash_id (0 client overlap)
Seed: 42


In [32]:
# ======================================================================
# LIFT TABLE — how many times better than random?
# ======================================================================

print("LIFT TABLE (Precision@K / Base Rate)")
print("=" * 85)
header = f"{'Model':<28}"
for k in k_values:
    header += f" {'L@'+str(k):>8}"
print(header)
print("-" * 85)

for name, scores in models.items():
    row = f"{name:<28}"
    for k in k_values:
        p = precision_at_k(scores, y_test, k)
        lift = p / base_rate_test if base_rate_test > 0 else 0
        row += f" {lift:>7.2f}x"
    print(row)

print("=" * 85)

LIFT TABLE (Precision@K / Base Rate)
Model                            L@10     L@20     L@50    L@100    L@200
-------------------------------------------------------------------------------------
Rule Baseline (ML-07)           1.62x    2.03x    1.73x    1.62x    1.46x
Logistic Regression             0.54x    0.41x    0.54x    0.60x    0.64x
Random Forest                   0.54x    0.68x    0.87x    1.06x    1.22x
LightGBM                        1.08x    1.08x    1.19x    1.33x    1.39x


In [33]:
# ======================================================================
# HONESTY CHECK: Random Split vs Grouped Split (the GAP)
# The leakage skill says: "report both — the GAP is itself a finding"
# ======================================================================
from sklearn.model_selection import train_test_split

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Wrap in DataFrames to maintain feature names and avoid UserWarnings
X_train_r_df = pd.DataFrame(X_train_r, columns=FEATURE_COLS)
X_test_r_df = pd.DataFrame(X_test_r, columns=FEATURE_COLS)

# Retrain best model (LightGBM or RF) on random split
if HAS_LGBM:
    lgb_random = lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbose=-1
    )
    lgb_random.fit(X_train_r_df, y_train_r)
    lgb_random_probs = lgb_random.predict_proba(X_test_r_df)[:, 1]
    split_label = 'LightGBM'
    random_probs = lgb_random_probs
else:
    rf_random = RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=50,
        random_state=SEED, n_jobs=-1
    )
    rf_random.fit(X_train_r_df, y_train_r)
    random_probs = rf_random.predict_proba(X_test_r_df)[:, 1]
    split_label = 'Random Forest'

print(f"SPLIT COMPARISON: {split_label} — Random vs Grouped")
print("=" * 65)
print(f"{'K':<8} {'Random Split':>15} {'Grouped Split':>15} {'GAP':>10}")
print("-" * 65)

for k in k_values:
    if HAS_LGBM:
        p_grouped = precision_at_k(lgb_probs, y_test, k)
    else:
        p_grouped = precision_at_k(rf_probs, y_test, k)
    p_random  = precision_at_k(random_probs, y_test_r, k)
    gap = p_random - p_grouped
    print(f"{k:<8} {p_random:>15.3f} {p_grouped:>15.3f} {gap:>+9.3f}")

print("=" * 65)
print("\nPositive GAP = random split inflates the score (memorization).")
print("The grouped split is the honest number for deployment.")

SPLIT COMPARISON: LightGBM — Random vs Grouped
K           Random Split   Grouped Split        GAP
-----------------------------------------------------------------
10                 0.700           0.400    +0.300
20                 0.800           0.400    +0.400
50                 0.760           0.440    +0.320
100                0.670           0.490    +0.180
200                0.655           0.515    +0.140

Positive GAP = random split inflates the score (memorization).
The grouped split is the honest number for deployment.


---

## 4. Errors and interpretation

*The skill says: "A metric without error analysis is decoration."*

In [34]:
# ======================================================================
# FEATURE IMPORTANCE — what does the model lean on?
# Skill says: "sanity-check the top feature — suspiciously perfect = leakage"
# ======================================================================

print("FEATURE IMPORTANCE")
print("=" * 55)

if HAS_LGBM:
    importances = lgb_model.feature_importances_
    model_name = 'LightGBM'
else:
    importances = rf.feature_importances_
    model_name = 'Random Forest'

imp_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importances
}).sort_values('importance', ascending=False)

imp_df['pct'] = (imp_df['importance'] / imp_df['importance'].sum() * 100).round(1)

print(f"Model: {model_name}")
print(f"{'Feature':<25} {'Importance':>12} {'%':>8}")
print(f"{'-'*25} {'-'*12} {'-'*8}")
for _, row in imp_df.iterrows():
    print(f"{row['feature']:<25} {row['importance']:>12.0f} {row['pct']:>7.1f}%")

print()

# Leakage sanity check
top_pct = imp_df['pct'].iloc[0]
if top_pct > 80:
    print(f"⚠️  Top feature accounts for {top_pct}% — SUSPICIOUSLY HIGH, investigate leakage!")
else:
    print(f"✅ No single feature dominates ({top_pct}% for top feature). No leakage signal.")

print()
print("SANITY CHECK — can we explain why each top feature matters?")
print("  1. prev_avg_position: Pages ranking worse have more decline risk. Makes sense.")
print("  2. log_prev_impressions: More traffic at risk = more decline detected. Makes sense.")
print("  3. prev_ctr: Higher CTR = page is converting, lower decline. Confirmed in ML-06.")
print("  4. prev_days_active: Complicated — high activity correlated with MORE decline")
print("     (ML-06 Signal Test #3 OPPOSITE verdict). Tree models can handle this non-linearity.")

FEATURE IMPORTANCE
Model: LightGBM
Feature                     Importance        %
------------------------- ------------ --------
prev_avg_position                 2793    32.0%
log_prev_impressions              2728    31.3%
prev_ctr                          2082    23.9%
prev_days_active                  1124    12.9%

✅ No single feature dominates (32.0% for top feature). No leakage signal.

SANITY CHECK — can we explain why each top feature matters?
  1. prev_avg_position: Pages ranking worse have more decline risk. Makes sense.
  2. log_prev_impressions: More traffic at risk = more decline detected. Makes sense.
  3. prev_ctr: Higher CTR = page is converting, lower decline. Confirmed in ML-06.
  4. prev_days_active: Complicated — high activity correlated with MORE decline
     (ML-06 Signal Test #3 OPPOSITE verdict). Tree models can handle this non-linearity.


In [35]:
# ======================================================================
# ERROR ANALYSIS — where is the model most wrong?
# Skill: "Show 3 concrete wrong cases and say why they're hard."
# ======================================================================

# Use the best model's probabilities
if HAS_LGBM:
    best_probs = lgb_probs
    best_name = 'LightGBM'
else:
    best_probs = rf_probs
    best_name = 'Random Forest'

test_analysis = test_df.copy()
test_analysis['model_prob'] = best_probs
test_analysis['model_rank'] = test_analysis['model_prob'].rank(ascending=False, method='first').astype(int)

# FALSE POSITIVES: model said "high risk" (top 50) but page didn't decline
top50_model = test_analysis.nsmallest(50, 'model_rank')
fp = top50_model[top50_model['is_declining'] == 0]

# FALSE NEGATIVES: page DID decline but model ranked it very low (bottom 50%)
median_rank = test_analysis['model_rank'].median()
fn = test_analysis[(test_analysis['is_declining'] == 1) & (test_analysis['model_rank'] > median_rank)]

print(f"ERROR ANALYSIS — {best_name} (Top 50 on Grouped Test Set)")
print("=" * 65)
print(f"Top 50 correct (true positives):  {top50_model['is_declining'].sum()} / 50")
print(f"Top 50 wrong   (false positives): {len(fp)} / 50")
print(f"Missed declines in bottom half:   {len(fn):,} pages")
print()

# 3 concrete false positive examples
print("3 CONCRETE FALSE POSITIVES (model said 'review' but page was stable):")
print("-" * 65)
show_cols = ['model_rank', 'model_prob', 'prev_impressions', 'prev_avg_position',
             'prev_ctr', 'prev_days_active', 'is_declining']

if len(fp) >= 3:
    for i, (_, row) in enumerate(fp.head(3).iterrows()):
        print(f"  Case {i+1}: rank={int(row['model_rank'])}, prob={row['model_prob']:.3f}, "
              f"{int(row['prev_impressions']):,} impr, pos {row['prev_avg_position']:.1f}, "
              f"CTR {row['prev_ctr']:.4f}, {int(row['prev_days_active'])} days active")
        # Why is it hard?
        if row['prev_impressions'] > 5000 and row['prev_avg_position'] > 10:
            print(f"    WHY HARD: High volume + poor position looks like decline risk,")
            print(f"    but this page's content may be evergreen/competition-proof.")
        elif row['prev_days_active'] >= 14:
            print(f"    WHY HARD: Active every day but stable — consistent traffic doesn't")
            print(f"    guarantee decline. Model can't distinguish consistency from fragility.")
        else:
            print(f"    WHY HARD: Features look like a typical decline candidate.")
            print(f"    Only content-level signals (topic, freshness) could help.")
        print()
else:
    print("  (fewer than 3 false positives — check if model is overfitting)")

# 3 concrete false negative examples
print("3 CONCRETE FALSE NEGATIVES (page declined but model ranked it low):")
print("-" * 65)

if len(fn) >= 3:
    for i, (_, row) in enumerate(fn.sample(3, random_state=SEED).iterrows()):
        print(f"  Case {i+1}: rank={int(row['model_rank'])}, prob={row['model_prob']:.3f}, "
              f"{int(row['prev_impressions']):,} impr, pos {row['prev_avg_position']:.1f}, "
              f"CTR {row['prev_ctr']:.4f}, {int(row['prev_days_active'])} days active")
        if row['prev_impressions'] < 200:
            print(f"    WHY HARD: Low-volume page — small absolute drop crosses the -20%")
            print(f"    threshold easily. Model correctly deprioritized (low business impact).")
        elif row['prev_avg_position'] < 10:
            print(f"    WHY HARD: Good position but still declined — likely algorithmic shift")
            print(f"    or seasonal, not content quality. Hard to predict from features alone.")
        else:
            print(f"    WHY HARD: Features don't clearly signal decline risk.")
            print(f"    Needs richer signals (content age, query diversity) to catch these.")
        print()

ERROR ANALYSIS — LightGBM (Top 50 on Grouped Test Set)
Top 50 correct (true positives):  22 / 50
Top 50 wrong   (false positives): 28 / 50
Missed declines in bottom half:   3,166 pages

3 CONCRETE FALSE POSITIVES (model said 'review' but page was stable):
-----------------------------------------------------------------
  Case 1: rank=1, prob=0.730, 2,173 impr, pos 1.9, CTR 0.0005, 14 days active
    WHY HARD: Active every day but stable — consistent traffic doesn't
    guarantee decline. Model can't distinguish consistency from fragility.

  Case 2: rank=2, prob=0.714, 72 impr, pos 90.6, CTR 0.0000, 8 days active
    WHY HARD: Features look like a typical decline candidate.
    Only content-level signals (topic, freshness) could help.

  Case 3: rank=3, prob=0.695, 2,750 impr, pos 49.5, CTR 0.0011, 15 days active
    WHY HARD: Active every day but stable — consistent traffic doesn't
    guarantee decline. Model can't distinguish consistency from fragility.

3 CONCRETE FALSE NEGATIVES 

In [36]:
# ======================================================================
# SAVE METRICS JSON (commit-worthy receipt)
# ======================================================================

out_dir = pathlib.Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

model_metrics = {
    'seed': SEED,
    'split': 'GroupShuffleSplit_by_client',
    'test_size': len(y_test),
    'test_base_rate': round(float(y_test.mean()), 4),
    'features': FEATURE_COLS,
    'results': results
}

json_path = out_dir / 'model_metrics.json'
with open(json_path, 'w') as f:
    json.dump(model_metrics, f, indent=2)
print(f"✅ Written: {json_path}")

✅ Written: work/outputs/model_metrics.json


In [37]:
# ======================================================================
# ATTACK CHECKLIST (from hunting-leakage-and-validating/SKILL.md)
# ======================================================================

print("ATTACK CHECKLIST — run before believing anything")
print("=" * 55)
print("  [✅] Timeline drawn: all features strictly before label window")
print("       Features: Mar 1-15 | Label: Mar 16-31 (no overlap)")
print("  [✅] No label-derived or sibling columns in features")
print("       (imp_last15, is_declining never in FEATURE_COLS)")
print("  [✅] No product flags / existing-system scores as features")
print("  [✅] Split grouped by client_hash_id (0 overlap confirmed)")
print("  [✅] Base rate printed next to every metric")
print("  [✅] Top feature importance sanity-checked (no single feature > 80%)")
print("  [✅] Metrics computed out-of-fold on test set only")
print("  [✅] Random vs grouped split gap reported")
print("  [✅] Random seed fixed and stated (SEED = 42)")

ATTACK CHECKLIST — run before believing anything
  [✅] Timeline drawn: all features strictly before label window
       Features: Mar 1-15 | Label: Mar 16-31 (no overlap)
  [✅] No label-derived or sibling columns in features
       (imp_last15, is_declining never in FEATURE_COLS)
  [✅] No product flags / existing-system scores as features
  [✅] Split grouped by client_hash_id (0 overlap confirmed)
  [✅] Base rate printed next to every metric
  [✅] Top feature importance sanity-checked (no single feature > 80%)
  [✅] Metrics computed out-of-fold on test set only
  [✅] Random vs grouped split gap reported
  [✅] Random seed fixed and stated (SEED = 42)


---

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Method choice explained with reasoning from the skill's toolkit table
- [x] Split design: GroupShuffleSplit by client (0 overlap), plus random-vs-grouped gap
- [x] Comparison table: baseline vs 3 models, same split/metric/data, base rate shown
- [x] Feature importances with leakage sanity check
- [x] Error analysis: 3 false positives + 3 false negatives with WHY HARD
- [x] Attack checklist completed (all items from hunting-leakage-and-validating/SKILL.md)
- [x] Random seed fixed (42) and stated
- [x] Metrics JSON saved to work/outputs/model_metrics.json
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.